# M3. 지하철 혼잡 시간표

서울 지하철 역별 · 시간대별 승하차 인원(2026년 1~9월, 6,830행)을 긴 표로 바꿔 월별 Parquet 로 저장하고, MinIO 에 올린 뒤 DuckDB 로 s3:// 를 바로 조회해 「시간대별 붐비는 역」 순위를 냅니다.

- 단계: 기초 미니 프로젝트 · 정형 · 예상 3~4시간
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 서울 열린데이터광장 — 서울시 지하철 호선별 역별 시간대별 승하차 인원 정보(OA-12252) 중 2026년 1~9월, 공공누리 1유형(출처표시)

**빈칸 노트북** — 코드 속 `___①___` 자리 14곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/M3/M3_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

출근길 8시, 어느 역에서 가장 많이 타고 어느 역에서 가장 많이 내릴까요?
서울시는 호선 · 역마다 한 달 동안 시간대별로 몇 명이 타고 내렸는지를 매달 공개합니다. 이 자료로 **「시간대별 붐비는 역」 순위표**를 만듭니다.

흐름은 4주차 레이크하우스의 축소판입니다.

```text
CSV(원본, 넓은 표) → pandas 긴 표 · 정제 → Parquet(월별 폴더) → MinIO 버킷 → DuckDB 로 s3:// 조회(Trino 자리)
```

다 끝내면 다음을 말할 수 있습니다.

- 같은 자료가 두 번 들어간 달을 어떻게 찾고 지웠는지
- 열 48개짜리 넓은 표를 왜 긴 표로 바꾸는지
- 오전 8시 「타는 역」 1위와 「내리는 역」 1위 — 주거지와 일터가 갈리는 모습
- 객체 저장소(MinIO)에 올린 파일을 SQL 엔진이 어떻게 바로 읽는지

원래 카드의 조회 엔진은 Trino 입니다. Trino 는 무거운 자바 서버라 Colab 에서 띄우기 어려워, 같은 SQL 을 쓰는 **DuckDB 의 httpfs(S3 읽기)** 로 대신합니다. 내 컴퓨터 Docker 에서 Trino 로 하는 방법은 7단계 Tip 에 적었습니다.

## 1단계 · 준비 — 라이브러리와 데이터

Colab 에 pandas · DuckDB · pyarrow 는 있고, MinIO 파이썬 클라이언트(`minio`)는 없을 수 있어 설치합니다(4주차 17장에서 쓴 것과 같은 패키지).

데이터는 서울 열린데이터광장 「서울시 지하철 호선별 역별 시간대별 승하차 인원 정보」(OA-12252)입니다. 이용허락은 공공누리 1유형(출처표시)입니다.
전체(2015년부터, 8만 5천여 행)에서 2026년 1~9월만 골라 원본 그대로(CP949, 열 52개) 압축해 두었습니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("minio", "minio==7.2.20")

import os, shutil, urllib.request
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

path = fetch("seoul_subway_hourly_202601_09.csv.gz")
print(path, round(os.path.getsize(path) / 1e6, 2), "MB")

> **확인 포인트** — `data/seoul_subway_hourly_202601_09.csv.gz 0.67 MB` 가 찍힙니다.

> **Aebon's Tip** — 서울 열린데이터광장의 이 데이터셋은 「Sheet」 화면에서 내려받기(CSV)를 누르면 전체가 한 파일로 옵니다(약 31MB). 최신 달을 보고 싶으면 그 파일을 받아 `사용월` 로 골라 쓰세요.
> 같은 자료를 OpenAPI(`CardSubwayTime`)로도 주지만, 공개 키 `sample` 로는 5행까지만 오므로 인증키를 받아야 합니다.

## 2단계 · 읽기 — 일부러 기본 키 검사에 걸리기

In [ ]:
raw = pd.read_csv(path, encoding="cp949", dtype={"사용월": ___①___})   # ① 202601 을 글자로
print(raw.shape)
print(raw.columns[:5].tolist(), "...", raw.columns[-2:].tolist())

> **확인 포인트** — `(6830, 52)` — 사용월 · 호선명 · 지하철역 + 「04시-05시 승차인원」부터 「03시-04시 하차인원」까지 48개 + 작업일자입니다.

In [ ]:
assert raw.shape == (6830, 52)

이 표에서 「한 행」은 **한 달 · 한 호선 · 한 역**이어야 합니다. 그 세 열이 기본 키입니다. 정말 그런지 검사해 봅니다.

In [ ]:
KEY = ["사용월", "호선명", "지하철역"]
try:
    assert not raw.___②___(KEY).any(), "기본 키가 겹칩니다"   # ② 앞에 나온 행과 같은가
except AssertionError as e:
    print("실패:", e)
    print(raw[raw.duplicated(KEY)].groupby("사용월").size())

3월과 7월에 같은 키가 600여 개씩 겹칩니다. 겹친 행이 **모든 열까지 똑같은지** 확인합니다. 똑같다면 같은 자료가 두 번 실린 것이니 하나만 남기면 됩니다.
(열 값이 다르다면 어느 쪽이 맞는지 따로 판단해야 합니다 — 그때는 지우면 안 됩니다.)

In [ ]:
print("키가 겹치는 행:", raw.duplicated(KEY).sum(), "/ 모든 열이 같은 행:", raw.duplicated().sum())

wide = raw.___③___().reset_index(drop=True)   # ③ 똑같은 행 하나만 남기기
print(len(wide), wide.groupby("사용월").size().to_dict())

> **확인 포인트** — 겹친 1,241행이 모두 열까지 똑같습니다. 지우고 나면 **5,589행**, 달마다 620~623개 역이 남습니다.

In [ ]:
assert raw.duplicated(KEY).sum() == raw.duplicated().sum() == 1241
assert len(wide) == 5589 and not wide.duplicated(KEY).any()

> **자주 겪는 오류** — 이 겹침을 지우지 않으면 3월 · 7월 승하차 인원이 정확히 두 배로 잡힙니다. 그래프로 보면 「3월에 승객이 폭증」처럼 보여 그럴듯한 오답이 됩니다.

## 3단계 · 넓은 표를 긴 표로

지금 표는 시간대가 **열**로 펼쳐진 넓은 표(열 48개)입니다. 「8시에 가장 붐빈 역」을 구하려면 SQL 에서 열 이름을 하나하나 써야 합니다.
시간대를 **값**으로 내리면(긴 표) `WHERE hour = 8` 한 줄로 됩니다. pandas 의 `melt` 가 이 일을 합니다(3주차 12장).

In [ ]:
slot_cols = [c for c in wide.columns if "시-" in c]          # 「04시-05시 승차인원」 같은 48개 열
long = wide.___④___(   # ④ 넓은 표를 긴 표로
    id_vars=["사용월", "호선명", "지하철역"],
    value_vars=slot_cols,
    var_name="slot",
    value_name="riders",
)
print(len(slot_cols), long.shape)
long.head(3)

`slot` 에는 「04시-05시 승차인원」 같은 글자가 들어 있습니다. 여기서 **시작 시각**과 **승차/하차**를 꺼냅니다.
먼저 일부러 실패해 봅니다 — 글자를 통째로 숫자로 바꿀 수는 없습니다.

In [ ]:
try:
    long["slot"].str[:6].astype(int)        # '04시-05' 는 숫자가 아닙니다
except ValueError as e:
    print("실패:", str(e)[:60])

정규식으로 필요한 조각만 꺼냅니다. `(\d+)` 는 숫자 덩어리, `(승차|하차)` 는 둘 중 하나입니다.

In [ ]:
part = long["slot"].str.___⑤___(r"^(\d+)시-\d+시 (승차|하차)인원$")   # ⑤ 정규식 괄호 부분을 열로 꺼내기
long["hour"] = part[0].astype(int)
long["kind"] = part[1].map({"승차": "on", "하차": "off"})
long = long.rename(columns={"사용월": "ym", "호선명": "line", "지하철역": "station"}).drop(columns="slot")

print(long.shape, long["hour"].nunique(), long["kind"].unique().tolist())
print("전체 승하차:", int(long["riders"].sum()))

> **확인 포인트** — **268,272행**(5,589 × 48), 시각 24가지, `on` · `off` 두 가지. 9개월 승하차 합계는 **3,885,615,191명**입니다.
> (한 사람이 타고 내리면 두 번 세므로 「사람 수」가 아니라 「개표 · 집표 횟수」입니다.)

In [ ]:
assert long.shape == (268272, 6) and long["hour"].nunique() == 24
assert int(long["riders"].sum()) == 3885615191

## 4단계 · Parquet 로 저장 — 월별 폴더

긴 표를 Parquet 로 저장하면서 **사용월(ym)로 폴더를 나눕니다**(파티셔닝, 4주차 17장). `ym=202601/`, `ym=202602/` … 이렇게 폴더 이름에 값을 담는 방식을 Hive 파티션이라고 합니다.
나중에 「9월만」 조회하면 엔진은 `ym=202609` 폴더만 엽니다.

다시 돌려도 같은 결과가 나오도록, 쓰기 전에 옛 폴더를 지우고 파일 이름을 정해 둡니다(`part-0.parquet`).

In [ ]:
OUT = "data/subway_hourly"
shutil.rmtree(OUT, ignore_errors=True)
long.to_parquet(OUT, partition_cols=["___⑥___"], index=False, basename_template="part-{i}.parquet")   # ⑥ 폴더를 나눌 열

files = sorted(os.path.join(d, f) for d, _, fs in os.walk(OUT) for f in fs)
print(len(files), files[:2])
print("Parquet 합계:", round(sum(os.path.getsize(f) for f in files) / 1e6, 2), "MB")

> **확인 포인트** — 파일 **9개**(달마다 하나), `data/subway_hourly/ym=202601/part-0.parquet` 처럼 생겼습니다.

In [ ]:
assert len(files) == 9 and files[0].endswith("ym=202601/part-0.parquet")

DuckDB 로 다시 읽어 봅니다. 파일 안에는 `ym` 열이 없습니다 — 폴더 이름에만 있습니다. 그래서 Hive 파티션 읽기를 끄면 `ym` 을 찾지 못합니다.

In [ ]:
try:
    duckdb.sql(f"SELECT ym, count(*) FROM read_parquet('{OUT}/*/*.parquet', hive_partitioning=false) GROUP BY ym").df()
except duckdb.Error as e:
    print("실패:", type(e).__name__, "—", str(e).splitlines()[0][:70])

back = duckdb.sql(f"""
    SELECT ym, count(*) AS n
    FROM read_parquet('{OUT}/*/*.parquet', hive_partitioning=___⑦___)   -- ⑦ 폴더 이름을 열로 읽기
    GROUP BY ym ORDER BY ym
""").df()
back.head(3)

> **확인 포인트** — 켜면 달마다 행 수가 나옵니다 — 1월 29,808행(621개 역 × 48)입니다.

In [ ]:
assert int(back["n"].sum()) == 268272 and int(back.iloc[0]["n"]) == 29808

## 5단계 · MinIO 띄우고 올리기

MinIO 는 S3 와 같은 말(API)을 쓰는 객체 저장소입니다(4주차 17장). 4주차에는 Docker 로 띄웠지만 Colab 에는 Docker 가 없어서 **실행 파일 하나를 받아 백그라운드로** 띄웁니다.

MinIO 회사의 커뮤니티판 배포는 2025년에 멈춰서(4주차 17장 참고) 공식 내려받기 주소가 더 이상 파일을 주지 않습니다. 4주차와 같은 포크 판(pgsty, `RELEASE.2026-08-04T00-00-00Z` — Docker 의 `pgsty/minio` 와 같은 판)의 실행 파일을 받고, 같이 공개된 **체크섬으로 파일이 온전한지** 확인합니다.

이미 MinIO 가 떠 있으면(내 컴퓨터에서 4주차 Docker MinIO 를 켜 둔 경우 등) 환경 변수 `MINIO_ENDPOINT` 에 주소를 넣어 그것을 씁니다.

In [ ]:
import platform, time, tarfile, hashlib

ENDPOINT = os.environ.get("MINIO_ENDPOINT")       # 예: 127.0.0.1:9000 — 비어 있으면 아래에서 직접 띄웁니다
USER, PASSWORD = "minioadmin", "minioadmin"

if not ENDPOINT:
    VER = "RELEASE.2026-08-04T00-00-00Z"
    osname = {"Linux": "linux", "Darwin": "darwin"}[platform.system()]
    arch = "arm64" if platform.machine() in ("arm64", "aarch64") else "amd64"
    tgz = f"minio_20260804000000.0.0_{osname}_{arch}.tar.gz"
    base = f"https://github.com/pgsty/silo/releases/download/{VER}/"
    os.makedirs("bin", exist_ok=True)
    if not os.path.exists("bin/minio"):
        urllib.request.urlretrieve(base + tgz, "bin/" + tgz)                       # 약 33MB
        sums = urllib.request.urlopen(base + "minio_20260804000000.0.0_checksums.txt").read().decode()
        want = [line.split()[0] for line in sums.splitlines() if line.endswith(tgz)][0]
        got = hashlib.___⑧___(open("bin/" + tgz, "rb").read()).hexdigest()   # ⑧ 체크섬 알고리즘
        assert got == want, "체크섬이 다릅니다 — 받은 파일이 깨졌습니다"
        with tarfile.open("bin/" + tgz) as t:
            t.extract("minio", "bin", filter="data")
    env = dict(os.environ, MINIO_ROOT_USER=USER, MINIO_ROOT_PASSWORD=PASSWORD)
    server = subprocess.___⑨___(   # ⑨ 끝나기를 기다리지 않고 띄우기
        ["bin/minio", "server", "minio-data", "--address", "127.0.0.1:9000"],
        env=env, stdout=open("minio.log", "w"), stderr=subprocess.STDOUT,
    )
    ENDPOINT = "127.0.0.1:9000"

for _ in range(30):                                  # 살아 있는지 확인(health 주소)
    try:
        if urllib.request.urlopen(f"http://{ENDPOINT}/minio/health/live", timeout=2).status == 200:
            break
    except Exception:
        time.sleep(1)
print("MinIO:", ENDPOINT, "준비됨")

> **자주 겪는 오류** — `subprocess.run` 으로 띄우면 서버가 끝날 때까지 셀이 멈춥니다. 서버는 `Popen` 으로 띄워 놓고 다음 셀로 넘어갑니다. 셀이 멈췄다면 정지 버튼을 누르고 `Popen` 으로 바꾸세요.

이제 올립니다. 먼저 일부러 실패 — 버킷을 만들기 전에 올려 봅니다.

In [ ]:
from minio import Minio
from minio.error import S3Error

mc = Minio(ENDPOINT, access_key=USER, secret_key=PASSWORD, secure=___⑩___)   # ⑩ http 로 접속(SSL 끄기)
BUCKET, PREFIX = "subway", "curated/subway_hourly"

try:
    mc.fput_object(bucket_name=BUCKET, object_name="test.txt", file_path=path)
except S3Error as e:
    print("실패:", e.code)

버킷을 만들고, 4단계 폴더 구조를 **그대로 키 이름으로** 올립니다. 객체 저장소에는 진짜 폴더가 없고 `curated/subway_hourly/ym=202601/part-0.parquet` 전체가 한 객체의 이름입니다(4주차 17장).

In [ ]:
if not mc.bucket_exists(bucket_name=BUCKET):
    mc.___⑪___(bucket_name=BUCKET)   # ⑪ 버킷 만들기

for f in files:
    key = PREFIX + "/" + os.path.relpath(f, OUT).replace(os.sep, "/")
    mc.fput_object(bucket_name=BUCKET, object_name=key, file_path=f)

objs = [o.object_name for o in mc.list_objects(bucket_name=BUCKET, prefix=PREFIX + "/", recursive=True)]
print(len(objs), objs[0])

> **확인 포인트** — 객체 **9개**, 첫 객체는 `curated/subway_hourly/ym=202601/part-0.parquet` 입니다. 파일 이름을 정해 두었으므로 다시 돌려도 덮어쓸 뿐 늘어나지 않습니다.

In [ ]:
assert len(objs) == 9 and objs[0] == "curated/subway_hourly/ym=202601/part-0.parquet"

## 6단계 · DuckDB 로 s3:// 바로 조회 — 시간대별 붐비는 역

Trino 가 하던 일 — 객체 저장소의 Parquet 를 내려받지 않고 SQL 로 바로 읽기 — 를 DuckDB 의 `httpfs` 확장으로 합니다(3주차 14장 DuckDB).
접속 정보는 `SECRET` 으로 한 번 등록합니다. MinIO 는 `버킷.주소` 대신 `주소/버킷` 꼴(path style)로 부릅니다.

먼저 일부러 틀린 비밀번호로 해 봅니다.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

def s3_secret(secret):
    con.execute(f"""
        CREATE OR REPLACE SECRET minio (
            TYPE s3, KEY_ID '{USER}', SECRET '{secret}', REGION 'us-east-1',
            ENDPOINT '{ENDPOINT}', URL_STYLE '___⑫___', USE_SSL false   -- ⑫ MinIO 주소 꼴
        )
    """)

SRC = f"read_parquet('s3://{BUCKET}/{PREFIX}/*/*.parquet', hive_partitioning=true)"

s3_secret("wrong-password")
err = ""
try:
    con.sql(f"SELECT count(*) FROM {SRC}").fetchall()
except duckdb.Error as e:
    err = str(e).splitlines()[0].split("(")[-1].rstrip(")")      # 오류 첫 줄의 괄호 안 — HTTP 상태
    print("실패:", err)

> **확인 포인트** — `HTTP 403 Forbidden` 이 납니다 — 서버는 살아 있지만 열쇠가 틀렸다는 뜻입니다. 404(없음) · 연결 거부(서버 꺼짐)와 구분하세요.

In [ ]:
s3_secret(PASSWORD)
n_s3 = con.sql(f"SELECT count(*) FROM {SRC}").fetchall()[0][0]
n_sep = con.sql(f"SELECT count(*) FROM {SRC} WHERE ym = ___⑬___").fetchall()[0][0]   # ⑬ 9월 폴더만
print("전체:", n_s3, "/ 9월만:", n_sep)

> **확인 포인트** — 전체 **268,272행** — 로컬 Parquet 와 같습니다. 9월만은 29,808행이고, 이때 DuckDB 는 `ym=202609` 폴더 하나만 엽니다.

In [ ]:
assert err == "HTTP 403 Forbidden"
assert n_s3 == 268272 and n_sep == 29808

이제 순위입니다. **시간대마다 따로** 순위를 매기므로 `rank() OVER (PARTITION BY hour ...)` 입니다(2주차 8장 윈도 함수).
같은 역 이름이 여러 호선에 있으므로(서울역은 5개 호선) 순위는 「호선 + 역」으로 매깁니다.

In [ ]:
def top_by_hour(kind, hours=(8, 18), k=3):
    return con.sql(f"""
        SELECT hour, line, station, riders, rnk FROM (
            SELECT hour, line, station, CAST(sum(riders) AS BIGINT) AS riders,
                   rank() OVER (PARTITION BY ___⑭___ ORDER BY sum(riders) DESC) AS rnk   -- ⑭ 시간대마다 따로
            FROM {SRC}
            WHERE kind = '{kind}'
            GROUP BY hour, line, station
        )
        WHERE rnk <= {k} AND hour IN {tuple(hours)}
        ORDER BY hour, rnk
    """).df()

on_top = top_by_hour("on")
off_top = top_by_hour("off")
print("타는 역"); print(on_top)
print("내리는 역"); print(off_top)

> **확인 포인트** — 오전 8시 **타는 역 1위는 2호선 신림(9개월 2,400,088명)**, **내리는 역 1위는 7호선 가산디지털단지(3,380,369명)** 입니다.
> 저녁 18시에는 타는 역 1위가 2호선 강남(2,933,912명)으로 바뀝니다 — 아침에 사람이 모이는 곳(일터)과 흩어지는 곳(주거지)이 갈립니다.

In [ ]:
r8_on, r8_off = on_top[on_top["hour"] == 8].iloc[0], off_top[off_top["hour"] == 8].iloc[0]
r18_on = on_top[on_top["hour"] == 18].iloc[0]
assert (r8_on["station"], int(r8_on["riders"])) == ("신림", 2400088)
assert (r8_off["line"], r8_off["station"], int(r8_off["riders"])) == ("7호선", "가산디지털단지", 3380369)
assert (r18_on["station"], int(r18_on["riders"])) == ("강남", 2933912)

## 7단계 · 혼잡 시간표 그래프

하루 중 언제 붐비는지 전체 모양을 봅니다. 시간대별 승하차 합계와, 8시 내리는 역 1~3위의 시간대별 하차 인원입니다.

In [ ]:
import matplotlib.pyplot as plt

by_hour = con.sql(f"""
    SELECT hour, sum(riders) AS riders FROM {SRC} GROUP BY hour
""").df()
by_hour["order"] = (by_hour["hour"] - 4) % 24          # 04시부터 다음 날 03시 순서로
by_hour = by_hour.sort_values("order")
print(by_hour.sort_values("riders", ascending=False).head(2)[["hour", "riders"]])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_hour["hour"].astype(str), by_hour["riders"] / 1e6)
ax.set_title("Seoul subway riders by hour (Jan-Sep 2026, on + off)")
ax.set_xlabel("hour (start)")
ax.set_ylabel("million")
ax.grid(axis="y", alpha=.3)
plt.show()

> **확인 포인트** — 가장 붐비는 시간대는 **18시**, 그다음이 8시입니다(출근보다 퇴근이 더 몰립니다).

In [ ]:
assert by_hour.sort_values("riders", ascending=False)["hour"].tolist()[:2] == [18, 8]

In [ ]:
top3 = off_top[off_top["hour"] == 8]["station"].tolist()
prof = con.sql(f"""
    SELECT station, hour, sum(riders) AS riders FROM {SRC}
    JOIN (SELECT line, station FROM off_top WHERE hour = 8) USING (line, station)
    WHERE kind = 'off'
    GROUP BY station, hour
""").df().pivot(index="hour", columns="station", values="riders")
prof = prof.loc[sorted(prof.index, key=lambda h: (h - 4) % 24), top3]    # 시간은 04시부터, 열은 순위 순서로
prof.index = prof.index.astype(str)                 # 글자로 바꿔야 23시 다음에 0시가 오른쪽으로 이어집니다

ax = (prof / 1e3).plot(figsize=(10, 4), marker="o")
ax.set_title("Alighting by hour - top 3 stations at 8am")
ax.set_xlabel("hour (start)")
ax.set_ylabel("thousand")
ax.legend(["rank 1", "rank 2", "rank 3"])
ax.grid(alpha=.3)
plt.show()
print(top3)

> **확인 포인트** — 세 역 모두 8시에 하차가 치솟고 나머지 시간은 낮습니다 — 출근 목적지형 역입니다. 범례의 rank 1~3 은 `top3` 에 찍힌 순서(가산디지털단지 · 역삼 · 을지로입구)입니다.

> **Aebon's Tip** — 내 컴퓨터의 4주차 레이크하우스(Docker: MinIO + Iceberg REST + `trinodb/trino:483`, 4주차 20장)에서는 같은 순위를 Trino 로 냅니다.
> 4단계 Parquet 를 Iceberg 표로 올린 뒤(4주차 19장처럼 Spark 나 pyiceberg 로) Trino CLI 에서 아래처럼 씁니다. `rank() OVER (PARTITION BY ...)` · `GROUP BY` · `WHERE ym = 202609` 는 그대로이고, 다른 곳은 「파일 경로 대신 **표 이름**(`카탈로그.스키마.표`)을 쓴다」는 것뿐입니다.
> `SELECT hour, line, station, riders FROM (SELECT hour, line, station, sum(riders) AS riders, rank() OVER (PARTITION BY hour ORDER BY sum(riders) DESC) AS rnk FROM iceberg.subway.hourly WHERE kind = 'off' GROUP BY hour, line, station) WHERE rnk <= 3 AND hour IN (8, 18) ORDER BY hour, rnk;`
> DuckDB 처럼 `read_parquet('s3://…')` 로 파일을 직접 여는 문법은 Trino 에 없습니다 — Trino 는 반드시 카탈로그(Iceberg · Hive)를 거쳐 표를 찾습니다(4주차 21장).

In [ ]:
if not os.environ.get("MINIO_ENDPOINT"):
    server.terminate()                                   # 이 노트북이 띄운 MinIO 끄기
con.close()

## 8단계 · 더 해 보기

- 승차와 하차를 더한 「역 전체 혼잡」으로 시간대별 1위를 다시 매겨 보세요. 환승역은 호선을 합칠지(역 이름으로 `GROUP BY`) 나눌지 정하고 이유를 적어 보세요.
- `ym` 별로 8시 하차 1위 역이 달마다 같은지 `rank() OVER (PARTITION BY ym ...)` 로 확인해 보세요.
- 같은 9호선 구간이 달마다 `9호선2단계` · `9호선2~3단계` 처럼 이름이 다르게 적혀 있는지 `SELECT DISTINCT line` 으로 찾아보세요. 이름이 바뀐 키를 맞추는 것도 정제입니다.
- 서울 열린데이터광장의 전체 파일(2015년~)을 받아 2019년(코로나 전)과 2026년의 8시 승하차를 비교해 보세요.

## 정답

① `str` — 202601 을 글자로
② `duplicated` — 앞에 나온 행과 같은가
③ `drop_duplicates` — 똑같은 행 하나만 남기기
④ `melt` — 넓은 표를 긴 표로
⑤ `extract` — 정규식 괄호 부분을 열로 꺼내기
⑥ `ym` — 폴더를 나눌 열
⑦ `true` — 폴더 이름을 열로 읽기
⑧ `sha256` — 체크섬 알고리즘
⑨ `Popen` — 끝나기를 기다리지 않고 띄우기
⑩ `False` — http 로 접속(SSL 끄기)
⑪ `make_bucket` — 버킷 만들기
⑫ `path` — MinIO 주소 꼴
⑬ `202609` — 9월 폴더만
⑭ `hour` — 시간대마다 따로